In [8]:
import os
import time
import json
import re
import requests
from bs4 import BeautifulSoup
from git import Repo

# ==================== CONFIGURATION ====================
OUTPUT_HTML_FILE = "index.html"

# 1. INDIVIDUAL PRODUCT PRICE TRACKING CONFIG
PRODUCT_TRACKING_LIST = {
    "buydeem_g553": {
        "name": "BUYDEEM G553 Food Steamer",
        "url": "https://us.buydeem.com/products/electric-food-steamer-g553",
        "baseline_price": 229.99
    },
    "Grape Water 300ml": {
        "name": "Grape Water 300ml",
        "url": "https://us.caudalie.com/p/320C-1/grape-water-moisturizing-face-mist.html?ODE4=Y29uZmlndXJhYmxlLzgxOC82MDcy",
        "baseline_price": 27.00
    }
}

# 2. PROMO CODES & SITE-WIDE DEALS CRAWLER CONFIG
# To add a new store, simply create a new dictionary item below!
DEALS_TRACKING_CONFIG = {
    "Caudalie": {
        "store_name": "Caudalie US",
        "homepage_url": "https://caudalie.com",
        "fallback_deals": [
            "Enjoy a free Ice Roller with your $140+ order. Code: FREEZE",
            "Free mini Resveratrol-Lift Trio on orders $130+. Code: LIFTED"
        ]
    },
    # --- TEMPLATE: COPY THIS BLOCK TO ADD MORE SITES ---
    # "BrandName": {
    #     "store_name": "Brand Name Store",
    #     "homepage_url": "https://example.com",
    #     "fallback_deals": ["Free standard shipping on orders over $50"]
    # }
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
}
# ========================================================

def fetch_price(url):
    """Universal crawler supporting JSON-LD, Meta, and raw layout text extraction."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        if response.status_code != 200:
            return None
        soup = BeautifulSoup(response.content, "html.parser")
        
        # Strategy 1: Look for application/ld+json structures
        json_scripts = soup.find_all("script", type="application/ld+json")
        for script in json_scripts:
            try:
                if script.string:
                    data = json.loads(script.string.strip())
                    items = data if isinstance(data, list) else [data]
                    for item in items:
                        if "offers" in item:
                            offers = item["offers"]
                            if isinstance(offers, dict) and "price" in offers:
                                return float(offers["price"])
                            elif isinstance(offers, list) and len(offers) > 0 and "price" in offers:
                                return float(offers["price"])
            except Exception:
                continue

        # Strategy 2: Check OpenGraph metadata tags
        price_meta = soup.find("meta", property="og:price:amount") or soup.find("meta", name="twitter:data1")
        if price_meta and price_meta.get("content"):
            return float(price_meta["content"].replace("$", "").strip())

        # Strategy 3: Target text matches
        raw_text_matches = soup.find_all(text=re.compile(r'\$\d+\.\d{2}'))
        for element in raw_text_matches:
            text_str = element.strip()
            match = re.search(r'\d+\.\d{2}', text_str)
            if match:
                return float(match.group())
        return None
    except Exception as e:
        print(f"Error checking {url}: {e}")
        return None

def fetch_site_deals(store_id, config):
    """Dynamically parses custom brand layouts for promotional rules and keywords."""
    print(f"🎨 Scanning {config['store_name']} for active promotions & coupon texts...")
    deals = []
    try:
        response = requests.get(config["homepage_url"], headers=HEADERS, timeout=15)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, "html.parser")
            
            # Universal pattern regex: targets common coupon indicators (e.g. Code, Promo, Save, Off, Free)
            promo_elements = soup.find_all(text=re.compile(r'(code:|code\s*:|\b[A-Z]{4,}\b|\$\d+\+\s*order|save\s*\d+%|off\s*your\s*order|free\s*gift)', re.IGNORECASE))
            
            seen_texts = set()
            for elem in promo_elements:
                clean_text = elem.strip()
                # Text sanity length filtering rules (prevents parsing page source artifacts)
                if 20 < len(clean_text) < 150 and clean_text not in seen_texts:
                    clean_text = clean_text.replace("**", "").replace("  ", " ")
                    seen_texts.add(clean_text)
                    deals.append(clean_text)
                    
        # Fallback configuration safeguard if site anti-bot firewalls trigger
        if not deals:
            deals = config["fallback_deals"]
        return deals
    except Exception:
        return config["fallback_deals"]

def push_to_github():
    """Pushes the updated index.html file to your public GitHub repo."""
    try:
        repo_path = os.getcwd()
        repo = Repo(repo_path)
        repo.git.add(A=True)
        repo.index.commit(f"Multi-site Deals & Price sync: {time.strftime('%Y-%m-%d %H:%M:%S')}")
        origin = repo.remote(name='origin')
        origin.push()
        print("🌍 Success! Aggregated dashboard deployed live to GitHub Pages.")
    except Exception as e:
        print(f"⚠️ Git Push Failed: {e}")

def run_single_check():
    print("🚀 Running automated multi-site parser update...")
    
    # 1. Loop through and process all configured site deals boxes dynamically
    all_deals_html_blocks = []
    for store_id, config in DEALS_TRACKING_CONFIG.items():
        active_promos = fetch_site_deals(store_id, config)
        promo_list_items = "".join([f"<li>{deal}</li>" for deal in active_promos])
        
        all_deals_html_blocks.append(f"""
        <div class="deals-box">
            <h3>✨ Active {config['store_name']} Coupons & Offers</h3>
            <ul>{promo_list_items}</ul>
        </div>
        """)

    # 2. Process product catalog list pricing
    report_items = []
    any_drops = False

    for key, item in PRODUCT_TRACKING_LIST.items():
        current_price = fetch_price(item["url"])
        baseline = item["baseline_price"]
        
        if current_price is None:
            status_td = "<td style='color: #7f8c8d; font-weight: bold;'>Error fetching price</td>"
        elif current_price < baseline:
            any_drops = True
            savings = baseline - current_price
            status_td = f"<td style='color: #27ae60; background-color: #e8f8f5; font-weight: bold;'>🚨 PRICE DROP: ${current_price:.2f} (Saved ${savings:.2f})</td>"
        else:
            status_td = f"<td style='color: #2c3e50;'>Normal (${current_price:.2f})</td>"
            
        report_items.append(f"""
            <tr>
                <td><b>{item['name']}</b></td>
                <td>${baseline:.2f}</td>
                {status_td}
                <td><a href='{item['url']}' target='_blank'>Shop Link ↗</a></td>
            </tr>
        """)

    # 3. Compile Master Dashboard UI
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>Price & Deal Tracker Dashboard</title>
        <style>
            body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; background-color: #f8f9fa; margin: 40px; color: #333; }}
            .container {{ max-width: 900px; margin: 0 auto; background: white; padding: 30px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); }}
            h1 {{ margin-top: 0; color: #2c3e50; font-size: 24px; }}
            .timestamp {{ color: #7f8c8d; font-size: 14px; margin-bottom: 25px; }}
            table {{ width: 100%; border-collapse: collapse; margin-top: 20px; }}
            th, td {{ padding: 12px 15px; text-align: left; border-bottom: 1px solid #e1e8ed; }}
            th {{ background-color: #f4f6f8; color: #5c7080; font-weight: 600; }}
            a {{ color: #3498db; text-decoration: none; font-weight: 500; }}
            a:hover {{ text-decoration: underline; }}
            .alert-banner {{ background-color: #e74c3c; color: white; padding: 15px; border-radius: 6px; font-weight: bold; margin-bottom: 25px; }}
            .deals-container {{ display: flex; flex-direction: column; gap: 15px; margin-bottom: 25px; }}
            .deals-box {{ background-color: #fcf3cf; border-left: 5px solid #f1c40f; padding: 20px; border-radius: 4px; }}
            .deals-box h3 {{ margin-top: 0; color: #b7950b; font-size: 16px; margin-bottom: 10px; }}
            .deals-box ul {{ margin: 0; padding-left: 20px; color: #7d6608; font-weight: 500; line-height: 1.6; font-size: 14px; }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>🛍️ Shared Price & Deal Monitor</h1>
            <div class="timestamp">Last Updated: {time.strftime('%Y-%m-%d %I:%M:%S %p')}</div>
            
            <!-- Live Active Promotions Stack -->
            <div class="deals-container">
                {"".join(all_deals_html_blocks)}
            </div>
            
            { '<div class="alert-banner">🔥 Price Drop Alert! One or more tracked items are on sale.</div>' if any_drops else '' }
            
            <table>
                <thead>
                    <tr>
                        <th>Product Name</th>
                        <th>Target Baseline</th>
                        <th>Current Status</th>
                        <th>Action</th>
                    </tr>
                </thead>
                <tbody>
                    {"".join(report_items)}
                </tbody>
            </table>
        </div>


_IncompleteInputError: incomplete input (3682582164.py, line 178)